In [13]:
import cv2 
import mediapipe as mp 
import numpy as np
mp_drawing=mp.solutions.drawing_utils
mp_pose=mp.solutions.pose 

In [14]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

In [15]:
for lndmrk in mp_pose.PoseLandmark:
    print(lndmrk)

PoseLandmark.NOSE
PoseLandmark.LEFT_EYE_INNER
PoseLandmark.LEFT_EYE
PoseLandmark.LEFT_EYE_OUTER
PoseLandmark.RIGHT_EYE_INNER
PoseLandmark.RIGHT_EYE
PoseLandmark.RIGHT_EYE_OUTER
PoseLandmark.LEFT_EAR
PoseLandmark.RIGHT_EAR
PoseLandmark.MOUTH_LEFT
PoseLandmark.MOUTH_RIGHT
PoseLandmark.LEFT_SHOULDER
PoseLandmark.RIGHT_SHOULDER
PoseLandmark.LEFT_ELBOW
PoseLandmark.RIGHT_ELBOW
PoseLandmark.LEFT_WRIST
PoseLandmark.RIGHT_WRIST
PoseLandmark.LEFT_PINKY
PoseLandmark.RIGHT_PINKY
PoseLandmark.LEFT_INDEX
PoseLandmark.RIGHT_INDEX
PoseLandmark.LEFT_THUMB
PoseLandmark.RIGHT_THUMB
PoseLandmark.LEFT_HIP
PoseLandmark.RIGHT_HIP
PoseLandmark.LEFT_KNEE
PoseLandmark.RIGHT_KNEE
PoseLandmark.LEFT_ANKLE
PoseLandmark.RIGHT_ANKLE
PoseLandmark.LEFT_HEEL
PoseLandmark.RIGHT_HEEL
PoseLandmark.LEFT_FOOT_INDEX
PoseLandmark.RIGHT_FOOT_INDEX


In [20]:
def draw_dotted_line(frame, point, start, end, line_color):
    """Draw a dotted vertical line"""
    x = point[0]
    for y in range(start, end, 10):
        if y + 5 < end:
            cv2.line(frame, (x, y), (x, y+5), line_color, 2)

In [22]:
# Colors
COLORS = {
    'blue': (0, 127, 255),
    'red': (255, 50, 50),
    'green': (0, 255, 127),
    'light_green': (100, 233, 127),
    'yellow': (255, 255, 0),
    'white': (255, 255, 255),
    'light_blue': (102, 204, 255)
}

In [33]:
cap = cv2.VideoCapture(0)

## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        frame_height, frame_width, _ = frame.shape
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
      
        # Make detection
        results = pose.process(image)
    
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            
            # Get coordinates (convert normalized to pixel coordinates)
            left_shoulder = [
                int(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x * frame_width),
                int(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y * frame_height)
            ]
            left_hip = [
                int(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x * frame_width),
                int(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y * frame_height)
            ]
            left_knee = [
                int(landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x * frame_width),
                int(landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y * frame_height)
            ]
            left_ankle=[
                int(landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].x * frame_width),
                int(landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].y * frame_height)
            ]
            
            # Calculate torso vertical angle using their logic
            # CORRECTED: The vertex (b) should be at the hip
            # Point on vertical line should be directly above/below the hip
            vertical_point_hip = np.array([left_hip[0], 0])  # Point above hip on vertical axis
            vertical_point_knee=np.array([left_knee[0],0])
            vertical_point_ankle=np.array([left_ankle[0],0])
            
            # CORRECTED ORDER: vertical_point -> hip (vertex) -> shoulder
            torso_angle = calculate_angle(vertical_point_hip, left_hip, left_shoulder)
            knee_angle=calculate_angle(vertical_point_knee,left_knee,left_hip)
            ankle_angle=calculate_angle(vertical_point_ankle,left_ankle,left_knee)
            
            
            # Draw dotted vertical line from hip,knee and ankle
            draw_dotted_line(image, left_hip, start=max(0, left_hip[1]-100), end=min(frame_height, left_hip[1]+50), line_color=COLORS['blue'])
            draw_dotted_line(image, left_knee, start=max(0, left_knee[1]-100), end=min(frame_height, left_knee[1]+50), line_color=COLORS['blue'])
            draw_dotted_line(image, left_ankle, start=max(0, left_ankle[1]-100), end=min(frame_height, left_ankle[1]+50), line_color=COLORS['blue'])
            # Determine multiplier for arc direction (like in their code)
            torso_multiplier = 1 if left_shoulder[0] > left_hip[0] else -1
            knee_multiplier = 1 if left_hip[0] > left_knee[0] else -1
            ankle_multiplier = 1 if left_knee[0] > left_ankle[0] else -1
            
            # Draw the arc to show the angle
            cv2.ellipse(image, tuple(left_hip), (40, 40), 
                       angle=0, startAngle=-90, endAngle=-90 + torso_multiplier * int(torso_angle), 
                       color=COLORS['white'], thickness=3, lineType=cv2.LINE_AA)
            cv2.ellipse(image, tuple(left_knee), (40, 40), 
                       angle=0, startAngle=-90, endAngle=-90 + knee_multiplier * int(torso_angle), 
                       color=COLORS['white'], thickness=3, lineType=cv2.LINE_AA)
            cv2.ellipse(image, tuple(left_ankle), (40, 40), 
                       angle=0, startAngle=-90, endAngle=-90 + ankle_multiplier * int(torso_angle), 
                       color=COLORS['white'], thickness=3, lineType=cv2.LINE_AA)
            
            
            # Draw the torso line (hip to shoulder), knee line (knee to hip), shin line (ankle to knee)
            cv2.line(image, tuple(left_hip), tuple(left_shoulder), COLORS['light_blue'], 4, lineType=cv2.LINE_AA)
            cv2.line(image, tuple(left_hip), tuple(left_knee), COLORS['light_blue'], 4, lineType=cv2.LINE_AA)
            cv2.line(image, tuple(left_knee), tuple(left_ankle), COLORS['light_blue'], 4, lineType=cv2.LINE_AA)
            
            # Draw landmark points
            cv2.circle(image, tuple(left_shoulder), 7, COLORS['yellow'], -1, lineType=cv2.LINE_AA)
            cv2.circle(image, tuple(left_hip), 7, COLORS['yellow'], -1, lineType=cv2.LINE_AA)
            cv2.circle(image, tuple(left_knee), 7, COLORS['yellow'], -1, lineType=cv2.LINE_AA)
            cv2.circle(image, tuple(left_ankle), 7, COLORS['yellow'], -1, lineType=cv2.LINE_AA)
            
            # Display the angle value near hip, knee and ankle
            cv2.putText(image, f'{int(torso_angle)}', 
                       (left_hip[0] + 15, left_hip[1]), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS['light_green'], 2, lineType=cv2.LINE_AA)
            cv2.putText(image, f'{int(knee_angle)}', 
                       (left_knee[0] + 15, left_knee[1]), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS['light_green'], 2, lineType=cv2.LINE_AA)
            cv2.putText(image, f'{int(ankle_angle)}', 
                       (left_ankle[0] + 15, left_ankle[1]), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS['light_green'], 2, lineType=cv2.LINE_AA)
            
            # Display angle at the screen
            cv2.putText(image, f'TORSO ANGLE: {int(torso_angle)} deg', 
                       (30, 40), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS['green'], 2, lineType=cv2.LINE_AA)
            cv2.putText(image, f'Knee ANGLE: {int(knee_angle)} deg', 
                       (930, 40), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS['green'], 2, lineType=cv2.LINE_AA)
            cv2.putText(image, f'Ankle ANGLE: {int(ankle_angle)} deg', 
                       (930, 240), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS['green'], 2, lineType=cv2.LINE_AA)
                       
        except Exception as e:
            pass
        
        # Render detections
        #mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                #mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                #mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))               
        
        cv2.imshow('Angle Measurement', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

c:\ProgramData\anaconda3\envs\media\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
